# 03. Análise exploratória

**Entrada:** `saida/base_modelagem.parquet` (notebook 02)
**Saídas:** figuras em `figuras/`

Conjuntos: treino = dias 1 a 19, validação = dias 20 a 23, teste = dias 24 a 29. As estatísticas por classe usam apenas o treino e a validação (dias 1 a 23), para não usar o teste na análise.

## 1. Configuração do ambiente

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, time
import numpy as np
import pandas as pd

# Pasta de trabalho no Google Drive (dados intermediários e figuras)
BASE = '/content/drive/MyDrive/artigo_predicao'
for d in ['ev_parts', 'uso_parts', 'saida', 'figuras']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

KEY = ['job_id', 'task_index']   # identificador de uma tarefa no trace
FRAC = 10                        # amostra: jobs com job_id % 10 == 0 (~10% dos jobs)

import matplotlib.pyplot as plt
import seaborn as sns
FIG = f'{BASE}/figuras'

base = pd.read_parquet(f'{BASE}/saida/base_modelagem.parquet')
base['conjunto'] = np.where(base.dia >= 24, 'teste', np.where(base.dia >= 20, 'validação', 'treino'))

## 2. Tamanho e taxa de falha por conjunto

In [ ]:
base.groupby('conjunto').falha.agg(['size', 'sum', 'mean'])

## 3. Taxa de falha por dia do trace

In [ ]:
d = base.groupby('dia').falha.mean() * 100
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(d.index, d.values, color='#4C72B0')
ax.axvline(19.5, ls='--', c='gray'); ax.axvline(23.5, ls='--', c='gray')
ax.set_xlabel('Dia do trace'); ax.set_ylabel('Falhas (%)')
fig.tight_layout(); fig.savefig(f'{FIG}/taxa_falha_dia.png', dpi=200)
print(d.round(2).to_string())

## 4. Concentração das falhas por job

In [ ]:
fj = base[base.falha == 1].groupby('job_id').size().sort_values(ascending=False)
print('jobs com falha:', len(fj), '| os 10 maiores concentram', round(fj.head(10).sum() / fj.sum(), 3), 'das falhas')
fj.head(10)

## 5. Valores ausentes de CPI e MAI e sua relação com a falha

In [ ]:
for c in ['cpi', 'mai']:
    print(c, base.groupby(base[c].isna()).falha.agg(['size', 'mean']).to_dict())

## 6. Medianas dos atributos por classe (dias 1 a 23)

In [ ]:
FEAT = ['priority','scheduling_class','cpu_req','ram_req','disk_req','cpu_rate','canonical_mem',
        'assigned_mem','unmapped_cache','total_cache','disk_space','cpi','mai','sampled_cpu',
        'max_mem','max_cpu','tempo_obs','n_janelas']
tr = base[base.dia <= 23]
tr.groupby('falha')[FEAT].median().T

## 7. Distribuição de atributos selecionados por classe

In [ ]:
sel = ['cpu_rate', 'canonical_mem', 'max_cpu', 'max_mem', 'cpi', 'tempo_obs']
am = tr.sample(200_000, random_state=42)
fig, axs = plt.subplots(2, 3, figsize=(10, 6))
for ax, c in zip(axs.flat, sel):
    sns.boxplot(data=am, x='falha', y=c, ax=ax, showfliers=False)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['FINISH', 'FAIL']); ax.set_xlabel(''); ax.set_title(c)
fig.tight_layout(); fig.savefig(f'{FIG}/atributos_por_classe.png', dpi=200)